In [4]:
%pip install "gymnasium" "stable-baselines3" "numpy" "matplotlib"

   ---------------------------------------- 0.0/187.6 kB ? eta -:--:--
   ---------------------------------------- 187.6/187.6 kB 5.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
    --------------------------------------- 1.6/124.1 MB 33.8 MB/s eta 0:00:04
   - -------------------------------------- 4.5/124.1 MB 48.3 MB/s eta 0:00:03
   -- ------------------------------------- 6.7/124.1 MB 53.8 MB/s eta 0:00:03
   -- ------------------------------------- 9.3/124.1 MB 54.0 MB/s eta 0:00:03
   --- ------------------------------------ 11.7/124.1 MB 59.5 MB/s eta 0:00:02
   ---- ----------------------------------- 14.6/124.1 MB 65.6 MB/s eta 0:00:02
   ----- ---------------------------------- 17.4/124.1 MB 59.5 MB/s eta 0:00:02
   ------ --------------------------------- 20.0/124.1 MB 59.5 MB/s eta 0:00:02
   ------- -------------------------------- 22.8/124.1 MB 59.5 MB/s eta 0:00:02
   -------- ------------------------------- 25.5/124.1 MB 65.6

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\python312\\share'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


#defining how the cabin evironment works
class CabinEnv(gym.Env):

    def __init__(self):
        super().__init__()

        #grid size - change this according to our ideas
        self.size = 8
        self.max_steps = 150

        # defining actions - we first have to state what actions it can perform
        self.action_space = spaces.Discrete(7)

        #TODO: define the reward system using a reward function

        # defining what the agent can observe - we have to define what the agent can see so it can make decisions based on that
        self.observation_space = spaces.Box(
            low=0,
            high=max(self.size, self.max_steps),
            shape=(4,),
            dtype=np.int32
        )

        # stating where the obstacles are located in the cabin (change it later if we want)
        self.obstacles = {
            (2, 2), (2, 3), (4, 3), (5, 3)
        }

        # stating where the target and exit are located in the cabin (change it later if we want)
        self.target = (5, 5)
        self.exit = (7, 7)

    #getting the observation of the agent and turning it to an np array
    def _get_obs(self):
        return np.array([
            self.x,
            self.y,
            int(self.inspected),
            self.max_steps - self.steps
        ], dtype=np.int32)

    # resetting the environment to its initial state
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.x, self.y = (0, 0)
        self.inspected = False
        self.steps = 0

        return self._get_obs(), {}

    # defining the step function which takes an action and returns the next state, reward, and whether the episode is done
    def step(self, action):
        self.steps += 1
        reward = -0.1

        # TODO: Implement the reward function based on the agent's actions and the environment state

        # Candidate movement
        moves = {
            0: (0, -1),  # up
            1: (0, 1),   # down
            2: (-1, 0),  # left
            3: (1, 0)    # right
        }

        if action in moves:
            dx, dy = moves[action]
            nx, ny = self.x + dx, self.y + dy

            # Stay in the cabin boundaries
            if not (0 <= nx < self.size and
                    0 <= ny < self.size):
                reward -= 2

            # Do not pass through obstacles
            elif (nx, ny) in self.obstacles:
                reward -= 2

            else:
                self.x, self.y = nx, ny

        # Inspect the target location
        elif action == 4:
            if (self.x, self.y) == self.target:
                if not self.inspected:
                    self.inspected = True
                    reward += 10
            else:
                reward -= 0.5

        # Finish only after completing the task
        # and reaching the designated exit
        terminated = (
            self.inspected
            and (self.x, self.y) == self.exit
        )

        #reward function is not yet implemented
        if terminated:
            reward += 50

        truncated = self.steps >= self.max_steps

        return (
            self._get_obs(),
            reward,
            terminated,
            truncated,
            {}
        )


In [ ]:
from gymnasium.utils.env_checker import check_env

#creating and testing the cabin env
env = CabinEnv()

check_env(env)
obs, info = env.reset(seed=42)

print("Observation shape:", obs.shape)
print("Available actions:", env.action_space)
print(env.render())


#TODO: train the agent using an RL algorithm (e.g., DQN, PPO) and evaluate its performance in the CabinEnv.
#TODO: Implement the reward function
#TODO: visually implement the environment using a library like Pygame or Matplotlib to visualize the agent's movements and interactions with the cabin.
